# 2 · CD4/CD8 classification at the repertoire levelThe core result. Each repertoire (a donor's cloud of clonotypes) is summarized into one **mean+cov**descriptor; repertoires are then classified CD4 vs CD8 with the reference-set-size retrieval metric(`ref_size_sweep_auroc`). No model is trained — the frozen foundation embeddings are reused.**Then four controls** ask whether the signal is real: depth (rarefaction), a second descriptor,a V-gene baseline, and per-cohort generalization.

## SetupLoad the embedded clouds (from notebook 01, then passed through the frozen model) and build onedescriptor per repertoire. Labels are FACS ground truth.

In [ ]:
import sys, re, osimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltREPERTOIRE_DIR = '../scripts/repertoire'          # rep_* library (referenced repo)CLOUDS_DIR     = '../data/clouds_embedded_TRB'    # embedded clouds (patient data, not in repo)LANDMARKS      = '../data/landmarks_beta.npz'sys.path.insert(0, REPERTOIRE_DIR)import rep_data, rep_descriptors as rd, rep_metrics as rmdef parse(s):    pid = re.match(r'(MS\d+|T1D\d+|HD_[A-Za-z]+)', s).group(1)    coh = 'HD' if pid.startswith('HD') else ('T1D' if pid.startswith('T1D') else 'MS')    cell = 'CD4' if 'CD4' in s else 'CD8'    return pid, coh, cellfiles = rep_data.cloud_files(CLOUDS_DIR); stems = sorted(files)print(len(stems), 'embedded repertoires')

In [ ]:
# one mean+cov descriptor per repertoireV, lab, coh, pat = [], [], [], []for s in stems:    Z, w, _ = rep_data.load_cloud(files[s], cols=('w_log',))    V.append(rd.mean_cov_weighted_np(Z, w))    p, c, cl = parse(s); pat.append(p); coh.append(c); lab.append(cl)V = np.vstack(V); lab = np.array(lab); coh = np.array(coh); pat = np.array(pat)print(f'{V.shape[0]} repertoires | CD4: {(lab=="CD4").sum()} | CD8: {(lab=="CD8").sum()}')

## 2.1 · Main result — separability vs reference-set size**Metric.** Pick `r` reference repertoires of one class; score every other repertoire by its max cosineto that set; AUROC of "is it that class?"; average over draws and classes. The curve rises with `r`.**Insight:** even a single reference repertoire separates CD4/CD8 at AUROC ~0.93; with 25 references itreaches ~0.99. Whole repertoires carry a clean, aggregable lineage signal.

In [ ]:
ref_sizes = [1, 2, 5, 10, 15, 20, 25]sweep = rm.ref_size_sweep_auroc(V, lab, ref_sizes, n_draws=100, seed=0)ys = [sweep[r] for r in ref_sizes]fig, ax = plt.subplots(figsize=(8.5, 5.5))ax.plot(ref_sizes, ys, marker='o', markersize=9, linewidth=2.5, color='#065A82',        markerfacecolor='#065A82', markeredgecolor='white', markeredgewidth=1.5, label='mean+cov')for x, y in zip(ref_sizes, ys):    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 12),                ha='center', fontsize=9, color='#21295C', fontweight='bold')ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)ax.text(ref_sizes[-1], 0.515, 'chance (0.5)', ha='right', fontsize=9, color='gray')ax.set_xlabel('Reference-set size  (number of reference repertoires)', fontsize=12, fontweight='bold')ax.set_ylabel('Mean ROC-AUC  (CD4 vs CD8)', fontsize=12, fontweight='bold')ax.set_title('CD4/CD8 separability vs reference-set size', fontsize=12)ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)ax.legend(loc='lower right', fontsize=10)plt.tight_layout(); plt.show()

## 2.2 · Control 1 — depth (rarefaction)CD8 clouds are shallower than CD4 (seen in notebook 01). Could the model be reading *depth* instead of*biology*? Rarefy every cloud to the common floor K and re-run, averaging over rarefaction seeds.**Insight:** rarefied ≈ full depth → the signal is **not** a depth artifact.

In [ ]:
sizes = [len(rep_data.load_cloud(files[s], cols=('w_log',))[0]) for s in stems]K = min(sizes)print('rarefying to K =', K)n_seeds = 5rare = {r: [] for r in ref_sizes}for seed in range(n_seeds):    Vr = np.vstack([rd.mean_cov_weighted_np(*rep_data.load_cloud(files[s], cap=K, seed=seed, cols=('w_log',))[:2])                    for s in stems])    sw = rm.ref_size_sweep_auroc(Vr, lab, ref_sizes, n_draws=100, seed=seed)    for r in ref_sizes: rare[r].append(sw[r])rare_mean = [np.mean(rare[r]) for r in ref_sizes]; rare_std = [np.std(rare[r]) for r in ref_sizes]fig, ax = plt.subplots(figsize=(8.5, 5.5))ax.plot(ref_sizes, ys, marker='o', markersize=8, lw=2.5, color='#065A82', label='full depth')ax.errorbar(ref_sizes, rare_mean, yerr=rare_std, marker='s', markersize=7, lw=2.5,            color='#C0392B', label=f'rarefied to K={K}', capsize=3)ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)ax.set_xlabel('Reference-set size  (number of reference repertoires)', fontsize=12, fontweight='bold')ax.set_ylabel('Mean ROC-AUC  (CD4 vs CD8)', fontsize=12, fontweight='bold')ax.set_title('Depth control: full vs rarefied (overlap = not a depth artifact)', fontsize=12)ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)ax.legend(loc='lower right', fontsize=10)plt.tight_layout(); plt.show()

## 2.3 · Control 2 — depth-only baselineHow much does cloud *size alone* separate the classes, with no embedding? Depth is 1-D so cosinedegenerates; instead take the AUROC of "depth predicts class" directly (CD8 is shallower → score = −n).**Insight:** the depth-only baseline is well below the foundation curve → the model adds real biologybeyond depth.

In [ ]:
from sklearn.metrics import roc_auc_scoren_clono = np.array(sizes)y_cd8 = (lab == 'CD8').astype(int)depth_only_auc = roc_auc_score(y_cd8, -n_clono)print(f'depth-only baseline: {depth_only_auc:.4f}')fig, ax = plt.subplots(figsize=(8.5, 5.5))ax.plot(ref_sizes, ys, marker='o', markersize=9, lw=2.5, color='#065A82', label='mean+cov (foundation)')for x, y in zip(ref_sizes, ys):    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0, 12),                ha='center', fontsize=9, color='#21295C', fontweight='bold')ax.axhline(depth_only_auc, color='#C0392B', ls='--', lw=1.8, alpha=0.8,           label=f'depth-only baseline ({depth_only_auc:.3f})')ax.axhline(0.5, color='gray', ls=':', lw=1, alpha=0.6)ax.set_xlabel('Reference-set size  (number of reference repertoires)', fontsize=12, fontweight='bold')ax.set_ylabel('Mean ROC-AUC  (CD4 vs CD8)', fontsize=12, fontweight='bold')ax.set_title('Foundation vs depth-only baseline', fontsize=12)ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)ax.legend(loc='lower right', fontsize=10)plt.tight_layout(); plt.show()

## 2.4 · Control 3 — second descriptor + V-gene baselineTwo extra curves on the same sweep: **occupancy** (a landmark histogram — an independent way tosummarize the cloud) and **V-gene usage** (a simple non-foundation baseline).**Insight:** occupancy tracks mean+cov (signal is not descriptor-specific). The foundation beats theV-gene baseline, most at low reference counts — it adds information beyond V-gene usage, though V-geneusage is itself a strong lineage cue.

In [ ]:
curves = {'mean+cov': ys}# occupancyif os.path.exists(LANDMARKS):    protos = np.load(LANDMARKS)['centroids']    Vo = np.vstack([rd.weighted_occupancy(*rep_data.load_cloud(files[s], cols=('w_log',))[:2], protos, tau=0.1)                    for s in stems])    sw_o = rm.ref_size_sweep_auroc(Vo, lab, ref_sizes, n_draws=100, seed=0)    curves['occupancy'] = [sw_o[r] for r in ref_sizes]# V-gene usagevg = [rep_data.load_cloud(files[s], cols=('w_log','v_gene')) for s in stems]vocab = rm.build_vgene_vocab([extra['v_gene'] for _, _, extra in vg])V_vg = np.vstack([rm.vusage_vector(extra['v_gene'], w, vocab) for _, w, extra in vg])sw_vg = rm.ref_size_sweep_auroc(V_vg, lab, ref_sizes, n_draws=100, seed=0)curves['V-gene usage'] = [sw_vg[r] for r in ref_sizes]fig, ax = plt.subplots(figsize=(9, 5.8))styles = {'mean+cov':('#065A82','o'), 'occupancy':('#1C7293','^'), 'V-gene usage':('#C0392B','s')}for name, yv in curves.items():    c, m = styles[name]    ax.plot(ref_sizes, yv, marker=m, markersize=8, lw=2.4, color=c, label=name)ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)ax.set_xlabel('Reference-set size  (number of reference repertoires)', fontsize=12, fontweight='bold')ax.set_ylabel('Mean ROC-AUC  (CD4 vs CD8)', fontsize=12, fontweight='bold')ax.set_title('Descriptor comparison + V-gene baseline', fontsize=12)ax.set_xticks(ref_sizes); ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25)ax.legend(loc='lower right', fontsize=10)plt.tight_layout(); plt.show()

## 2.5 · Control 4 — does it generalize across cohorts?Run the sweep within each cohort (HD / MS / T1D) separately.**Insight:** all three rise together → the signal is homogeneous, not driven by one cohort. (The T1Dcurve is shorter and noisier — only 7+7 repertoires.)

In [ ]:
colors = {'ALL':'#065A82', 'HD':'#2C7FB8', 'MS':'#D95F02', 'T1D':'#7570B3'}fig, ax = plt.subplots(figsize=(9, 5.8))for c in ['ALL', 'HD', 'MS', 'T1D']:    if c == 'ALL':        Vc, lc = V, lab    else:        m = coh == c; Vc, lc = V[m], lab[m]    n_min = min((lc=='CD4').sum(), (lc=='CD8').sum())    rs = [r for r in ref_sizes if r < n_min]    sw = rm.ref_size_sweep_auroc(Vc, lc, rs, n_draws=100, seed=0)    style = dict(marker='o', lw=2.5) if c=='ALL' else dict(marker='s', lw=2, alpha=0.85)    ax.plot(rs, [sw[r] for r in rs], color=colors[c], label=f'{c} (n={len(Vc)})', **style)ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.6)ax.set_xlabel('Reference-set size  (number of reference repertoires)', fontsize=12, fontweight='bold')ax.set_ylabel('Mean ROC-AUC  (CD4 vs CD8)', fontsize=12, fontweight='bold')ax.set_title('Separability by cohort', fontsize=12)ax.set_ylim(0.45, 1.02); ax.grid(True, alpha=0.25); ax.legend(loc='lower right', fontsize=10)plt.tight_layout(); plt.show()

## Summary| Check | Result ||---|---|| Repertoires separate CD4/CD8? | **Yes** — AUROC ~0.93 (r=1) → ~0.99 (r=25) || Depth artifact? | **No** — rarefied ≈ full depth || Depth-only baseline | well below the foundation curve || Descriptor-specific? | **No** — occupancy tracks mean+cov || Beyond V-gene usage? | **Yes** — foundation > V-gene, esp. at low r || Cohort-driven? | **No** — holds within HD, MS, T1D |**Takeaway.** The frozen clonotype model carries a CD4/CD8 signal that **aggregates coherently to therepertoire level** and survives every standard confound. Next: is the descriptor *compositional* enoughto read CD4/CD8 fractions from mixtures? → notebook 04 (deconvolution).